In [1]:
import os, warnings, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import soxr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED      = 42
TARGET_SR = 32_000
AUDIO_LEN = 160_000   # 5 s @ 32 kHz

np.random.seed(SEED)
print("Imports OK")

Imports OK


In [2]:
# Master metadata (all datasets, all windows)
META_ALL_PATH = Path('/data2/mromaniuc/cet-det/cet_perchv2/meta_train.parquet')
meta = pd.read_parquet(META_ALL_PATH)
print("Metadata loaded")
meta.head

Metadata loaded


<bound method NDFrame.head of        audio_row                                          group_key  \
0              0  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA...   
1              1  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA...   
2              2  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA...   
3              3  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA...   
4              4  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA...   
...          ...                                                ...   
11764      11764                  MONISH_Bottlenose_Dolphin_016.wav   
11765      11765                  MONISH_Bottlenose_Dolphin_017.wav   
11766      11766                  MONISH_Bottlenose_Dolphin_018.wav   
11767      11767                  MONISH_Bottlenose_Dolphin_018.wav   
11768      11768                  MONISH_Bottlenose_Dolphin_019.wav   

            dataset        coarse_class  global_idx  \
0      Adriatic_Sea          background         0.0   
1      

In [3]:
# ── Inputs ────────────────────────────────────────────────────────────────────
META_ALL_PATH = Path('/data2/mromaniuc/cet-det/alltogether/full_exploration/dim_red/projector_input/meta_all.parquet')

# ── Outputs ───────────────────────────────────────────────────────────────────
OUT_DIR            = Path('/data2/mromaniuc/cet-det/tursiops_perch')
AUDIO_CACHE        = OUT_DIR / 'X_audio.npy'
META_OUT           = OUT_DIR / 'meta_train.parquet'
META_SELECTION_OUT = OUT_DIR / 'meta_train_selection.parquet'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Target species ────────────────────────────────────────────────────────────
TARGET_SPECIES = 'Tursiops_truncatus'
KEEP_CLASSES   = {TARGET_SPECIES, 'background'}

# ── Dataset root folders ──────────────────────────────────────────────────────
DATASET_ROOTS = {
    'Adriatic_Sea'      : Path('/data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA/Audio_data'),
    'ALNITAK_CAVANILLES': Path('/data2/mromaniuc/cet-det/datasets/ALNITAK_CAVANILLES'),
    'DCLDE_2026'        : Path('/data2/mromaniuc/cet-det/datasets/DCLDE_2026/audio'),
    'DOLPHINFREE'       : Path('/data2/mromaniuc/cet-det/datasets/DOLPHINFREE/Single_hydrophone/inputs/Audio_data'),
    'DRYAD'             : Path('/data2/mromaniuc/cet-det/datasets/DRYAD/Audio'),
    'ECOSS_testtrain'   : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/testingtraining_sounds'),
    'ECOSS_annot'       : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/annotated_sounds'),
    'ECOSS_enhanced'    : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/enhanced4AI_sounds'),
    'OLTREMARE'         : Path('/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS'),
    'FREMANTLE'         : None,
    'MONISH'            : None,
    'WATKINS'           : None,

}

# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"META_ALL_PATH : {'OK' if META_ALL_PATH.exists() else 'MISSING'}  {META_ALL_PATH}")
print(f"OUT_DIR       : {OUT_DIR}")
print(f"Target        : {TARGET_SPECIES} vs background")
print()
for ds, root in DATASET_ROOTS.items():
    if root is None:
        print(f"  {ds:20s}  Pattern-B (skip here)")
    else:
        status = 'OK' if root.exists() else 'MISSING'
        print(f"  {ds:20s}  {status}  {root}")


META_ALL_PATH : OK  /data2/mromaniuc/cet-det/alltogether/full_exploration/dim_red/projector_input/meta_all.parquet
OUT_DIR       : /data2/mromaniuc/cet-det/tursiops_perch
Target        : Tursiops_truncatus vs background

  Adriatic_Sea          OK  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA/Audio_data
  ALNITAK_CAVANILLES    OK  /data2/mromaniuc/cet-det/datasets/ALNITAK_CAVANILLES
  DCLDE_2026            OK  /data2/mromaniuc/cet-det/datasets/DCLDE_2026/audio
  DOLPHINFREE           OK  /data2/mromaniuc/cet-det/datasets/DOLPHINFREE/Single_hydrophone/inputs/Audio_data
  DRYAD                 OK  /data2/mromaniuc/cet-det/datasets/DRYAD/Audio
  ECOSS_testtrain       OK  /data2/mromaniuc/cet-det/datasets/ECOSS/testingtraining_sounds
  ECOSS_annot           OK  /data2/mromaniuc/cet-det/datasets/ECOSS/annotated_sounds
  ECOSS_enhanced        OK  /data2/mromaniuc/cet-det/datasets/ECOSS/enhanced4AI_sounds
  OLTREMARE             OK  /data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS

In [4]:
print('Loading master metadata...')
meta_all = pd.read_parquet(META_ALL_PATH)
print(f'  meta_all: {len(meta_all):,} rows\n')

def sample_n(df, n, seed=SEED):
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).copy()

# ── Find datasets that contain Tursiops truncatus ─────────────────────────────
tursiops_datasets = set(
    meta_all[meta_all['label_t4'] == TARGET_SPECIES]['dataset'].unique()
)

# Watkins and MONISH are Pattern-B (pickle-based) — handled separately later
tursiops_datasets -= {'WATKINS', 'MONISH', 'FREMANTLE'}

print(f'Datasets with {TARGET_SPECIES}:')
for ds in sorted(tursiops_datasets):
    n_sp = (meta_all[(meta_all['dataset'] == ds) &
                     (meta_all['label_t4'] == TARGET_SPECIES)]).shape[0]
    n_bg = (meta_all[(meta_all['dataset'] == ds) &
                     (meta_all['label_t2'].isin(['background','anthropogenic','geophonic']))]).shape[0]
    print(f'  {ds:25s}  {TARGET_SPECIES}: {n_sp:>5,}   background: {n_bg:>6,}')

# ── Keep only Tursiops + background rows from those datasets ─────────────────
# Background: label_t2 in background/anthropogenic/geophonic (matches get_coarse logic)
is_target  = meta_all['label_t4'] == TARGET_SPECIES
is_bg      = meta_all['label_t2'].isin(['background', 'anthropogenic', 'geophonic'])
in_dataset = meta_all['dataset'].isin(tursiops_datasets)

meta_kept = meta_all[in_dataset & (is_target | is_bg)].copy().reset_index(drop=True)

print(f'\nFiltered to {len(meta_kept):,} rows')
print('\nPer-dataset breakdown:')
print(meta_kept.groupby(['dataset', 'label_t2']).size().unstack(fill_value=0).to_string())

# Restrict DATASET_ROOTS to only relevant datasets
DATASET_ROOTS = {k: v for k, v in DATASET_ROOTS.items() if k in tursiops_datasets}


Loading master metadata...
  meta_all: 247,630 rows

Datasets with Tursiops_truncatus:
  ALNITAK_CAVANILLES         Tursiops_truncatus:    58   background:  4,348
  Adriatic_Sea               Tursiops_truncatus:   942   background:    238
  DRYAD                      Tursiops_truncatus: 2,916   background:  3,854
  ECOSS_enhanced             Tursiops_truncatus:    27   background:     25
  ECOSS_testtrain            Tursiops_truncatus:    19   background: 35,917
  OLTREMARE                  Tursiops_truncatus: 2,904   background: 13,956

Filtered to 65,204 rows

Per-dataset breakdown:
label_t2            anthropogenic  background  odontocete
dataset                                                  
ALNITAK_CAVANILLES              0        4348          58
Adriatic_Sea                    0         238         942
DRYAD                           0        3854        2916
ECOSS_enhanced                 25           0          27
ECOSS_testtrain             33585        2332          19
OL

In [6]:
# Binary coarse_class: Tursiops_truncatus or background
def get_coarse(row):
    t4 = row.get('label_t4', None)
    if pd.notna(t4) and str(t4).strip() == TARGET_SPECIES:
        return TARGET_SPECIES
    return 'background'

meta_kept['coarse_class'] = meta_kept.apply(get_coarse, axis=1)

print('Coarse class distribution:')
for cls, n in meta_kept['coarse_class'].value_counts().items():
    print(f'  {cls:40s}  {n:>6,}')
print(f'\nClass ratio (bg:target): '
      f"{(meta_kept['coarse_class']=='background').sum()} : "
      f"{(meta_kept['coarse_class']==TARGET_SPECIES).sum()}")

assert (meta_kept['coarse_class'] == 'unknown').sum() == 0

meta_kept.to_parquet(META_SELECTION_OUT, index=False)
print(f'\nWritten: {META_SELECTION_OUT}')


Coarse class distribution:
  background                                58,338
  Tursiops_truncatus                         6,866

Class ratio (bg:target): 58338 : 6866

Written: /data2/mromaniuc/cet-det/tursiops_perch/meta_train_selection.parquet


In [7]:
import pickle

WATKINS_PKL = Path('/data2/mromaniuc/cet-det/models/perch_v2/WATKINS/windows_df.pkl')
MONISH_PKL  = Path('/data2/mromaniuc/cet-det/models/perch_v2/MONISH/models/perch_v2/LAST/windows_df.pkl')

with open(WATKINS_PKL, 'rb') as f:
    watkins_df = pickle.load(f)
with open(MONISH_PKL, 'rb') as f:
    monish_df  = pickle.load(f)

watkins_df['dataset'] = 'WATKINS'
monish_df['dataset']  = 'MONISH'

pattern_b_all = pd.concat([watkins_df, monish_df], ignore_index=True)
pattern_b_all['coarse_class'] = pattern_b_all['label']
pattern_b_all['group_key'] = pattern_b_all['dataset'] + '_' + pattern_b_all['source_file']

# Keep only Tursiops from datasets that have it
# (Watkins has 24 TT, MONISH has 28 TT per earlier table)
pb_tursiops_datasets = set(
    pattern_b_all[pattern_b_all['label'] == TARGET_SPECIES]['dataset'].unique()
)
print(f'Pattern-B datasets with {TARGET_SPECIES}: {pb_tursiops_datasets}')

pattern_b = pattern_b_all[
    pattern_b_all['dataset'].isin(pb_tursiops_datasets) &
    (pattern_b_all['label'] == TARGET_SPECIES)
].reset_index(drop=True)
# Note: no background rows in Watkins/MONISH for Tursiops datasets,
# so we only take the positive examples.

print(f'\nPattern-B Tursiops rows: {len(pattern_b):,}')
print(pattern_b.groupby(['dataset', 'label']).size().to_string())


Pattern-B datasets with Tursiops_truncatus: {'MONISH', 'WATKINS'}

Pattern-B Tursiops rows: 52
dataset  label             
MONISH   Tursiops_truncatus    28
WATKINS  Tursiops_truncatus    24


In [8]:
# ── 1. Load existing meta_selection and append Pattern-B metadata ─────────────
meta_selection = pd.read_parquet(META_SELECTION_OUT)
print(f"meta_selection before: {len(meta_selection):,} rows")

# Build minimal metadata rows for pattern-B (no wav_path — audio is in pickle)
pb_meta = pattern_b[['dataset', 'coarse_class', 'group_key', 'label',
                      'source_file', 'window_idx', 'window_kind']].copy()

# Add columns present in meta_selection but absent in pattern-B (fill with None)
for col in meta_selection.columns:
    if col not in pb_meta.columns:
        pb_meta[col] = None

# Reorder to match
pb_meta = pb_meta[meta_selection.columns]

meta_combined = pd.concat([meta_selection, pb_meta], ignore_index=True)
print(f"meta_selection after : {len(meta_combined):,} rows")

print("\nFull coarse_class distribution:")
for cls, n in meta_combined['coarse_class'].value_counts().items():
    print(f"  {cls:40s}  {n:>6,}")

print("\nFull dataset distribution:")
for ds, n in meta_combined['dataset'].value_counts().items():
    print(f"  {ds:25s}  {n:>6,}")

meta_selection before: 65,204 rows
meta_selection after : 65,256 rows

Full coarse_class distribution:
  background                                58,338
  Tursiops_truncatus                         6,918

Full dataset distribution:
  ECOSS_testtrain            35,936
  OLTREMARE                  16,860
  DRYAD                       6,770
  ALNITAK_CAVANILLES          4,406
  Adriatic_Sea                1,180
  ECOSS_enhanced                 52
  MONISH                         28
  WATKINS                        24


In [9]:
# ── 2. Build X_audio.npy ──────────────────────────────────────────────────────
import librosa, soxr
from tqdm.auto import tqdm

AUDIO_CACHE = OUT_DIR / 'X_audio.npy'
N           = len(meta_combined)

print(f"Allocating X_audio: ({N}, {AUDIO_LEN}) float32  "
      f"= {N * AUDIO_LEN * 4 / 1e9:.2f} GB")

# Open in r+ if exists (resume), w+ if new
cache_exists = Path(AUDIO_CACHE).exists()
mode = 'r+' if cache_exists else 'w+'
X_audio = np.lib.format.open_memmap(
    str(AUDIO_CACHE), mode=mode, dtype=np.float32, shape=(N, AUDIO_LEN)
)

if cache_exists:
    # A row is "done" if it has any non-zero value
    done_mask = np.any(X_audio != 0, axis=1)
    n_already = done_mask.sum()
    print(f"Resuming: {n_already:,} rows already filled, {N - n_already:,} to go")
else:
    done_mask = np.zeros(N, dtype=bool)
    print("Fresh cache — processing all rows")

# Build a lookup for pattern-B audio: (dataset, source_file, window_idx) → audio
pb_lookup = {
    (row['dataset'], row['source_file'], row['window_idx']): row['audio']
    for _, row in pattern_b.iterrows()
}

errors = []
for i, row in tqdm(meta_combined.iterrows(), total=N, desc='Building cache'):
    if done_mask[i]:
        continue
    try:
        ds = row['dataset']
        if ds in ('WATKINS', 'MONISH'):
            key = (ds, row['source_file'], row['window_idx'])
            X_audio[i] = pb_lookup[key]
        else:
            wav  = row['wav_path']
            off  = float(row['offset_s']) if pd.notna(row['offset_s']) else 0.0
            audio_native, native_sr = librosa.load(
                wav, sr=None, offset=off, duration=5.0, mono=True)
            if native_sr != TARGET_SR:
                audio = soxr.resample(audio_native, native_sr, TARGET_SR)
            else:
                audio = audio_native
            if len(audio) < AUDIO_LEN:
                audio = np.pad(audio, (0, AUDIO_LEN - len(audio)))
            X_audio[i] = audio[:AUDIO_LEN].astype(np.float32)
    except Exception as e:
        errors.append((i, row.get('dataset'), row.get('wav_path'), str(e)))
        X_audio[i] = np.zeros(AUDIO_LEN, dtype=np.float32)

X_audio.flush()
print(f"\nDone.  {N - len(errors):,} OK  |  {len(errors)} errors")
if errors:
    print("\nFirst 10 errors:")
    for i, ds, wav, msg in errors[:10]:
        print(f"  row={i}  {ds}  {wav}  →  {msg}")

# ── 3. Add audio_row and save final meta_train.parquet ────────────────────────
meta_combined = meta_combined.reset_index(drop=True)
meta_combined['audio_row'] = meta_combined.index.astype(np.int64)

META_TRAIN_OUT = OUT_DIR / 'meta_train.parquet'
meta_combined.to_parquet(META_TRAIN_OUT, index=False)

assert len(meta_combined) == X_audio.shape[0]
assert (meta_combined['audio_row'].values == np.arange(len(meta_combined))).all()

print(f"\nWritten: {META_TRAIN_OUT}")
print(f"  Rows          : {len(meta_combined):,}")
print(f"  X_audio shape : {X_audio.shape}")
print(f"\nReady for cet_perch_finetune_run.ipynb ✓")

Allocating X_audio: (65256, 160000) float32  = 41.76 GB
Resuming: 65,256 rows already filled, 0 to go


Building cache:   0%|          | 0/65256 [00:00<?, ?it/s]


Done.  65,256 OK  |  0 errors

Written: /data2/mromaniuc/cet-det/tursiops_perch/meta_train.parquet
  Rows          : 65,256
  X_audio shape : (65256, 160000)

Ready for cet_perch_finetune_run.ipynb ✓


In [10]:
print("Validating cache...")
X_audio_r = np.load(str(AUDIO_CACHE), mmap_mode='r')
print(f"  Shape  : {X_audio_r.shape}")
print(f"  Dtype  : {X_audio_r.dtype}")
print(f"  Global min/max: {X_audio_r.min():.4f} / {X_audio_r.max():.4f}")

print("\nPer-class amplitude stats (RMS):")
for cls in sorted(meta_combined['coarse_class'].unique()):
    idx = meta_combined[meta_combined['coarse_class'] == cls].index.tolist()
    sample_idx = idx[:min(50, len(idx))]
    rms_vals = np.sqrt(np.mean(X_audio_r[sample_idx] ** 2, axis=1))
    print(f"  {cls:40s}  n={len(idx):>5,}  "
          f"rms_mean={rms_vals.mean():.5f}  rms_std={rms_vals.std():.5f}")

zero_rows = np.where(~X_audio_r.any(axis=1))[0]
print(f"\nAll-zero rows (failed loads): {len(zero_rows)}")
if len(zero_rows) > 0:
    print(f"  Row indices: {zero_rows[:20]}")

Validating cache...
  Shape  : (65256, 160000)
  Dtype  : float32
  Global min/max: -1.6851 / 1.7455

Per-class amplitude stats (RMS):
  Tursiops_truncatus                        n=6,918  rms_mean=0.04462  rms_std=0.01214
  background                                n=58,338  rms_mean=0.03826  rms_std=0.01048

All-zero rows (failed loads): 0


## Write `meta_train.parquet`

In [11]:
# group_key is missing from meta_combined — derive it
def derive_group_key(row):
    # Pattern-B (Watkins/MONISH) — already have source_file
    if row.get('dataset') in ('WATKINS', 'MONISH'):
        return f"{row['dataset']}_{row['source_file']}"
    # Pattern-A — use wav_path as unique file identifier
    wp = row.get('wav_path', None)
    if pd.notna(wp) and str(wp).strip() not in ('', 'None'):
        return str(wp).strip()
    # Fallback
    return f"{row.get('dataset','')}_{row.get('global_idx','')}"

meta_combined['group_key'] = meta_combined.apply(derive_group_key, axis=1)

print(f"Unique group_keys: {meta_combined['group_key'].nunique():,}  "
      f"(out of {len(meta_combined):,} windows)")
print("\nGroups per dataset:")
for ds, grp in meta_combined.groupby('dataset'):
    print(f"  {ds:25s}  {grp['group_key'].nunique():>5,} groups  {len(grp):>6,} windows")

Unique group_keys: 1,500  (out of 65,256 windows)

Groups per dataset:
  ALNITAK_CAVANILLES           390 groups   4,406 windows
  Adriatic_Sea                   1 groups   1,180 windows
  DRYAD                         58 groups   6,770 windows
  ECOSS_enhanced                48 groups      52 windows
  ECOSS_testtrain              679 groups  35,936 windows
  MONISH                        19 groups      28 windows
  OLTREMARE                    281 groups  16,860 windows
  WATKINS                       24 groups      24 windows


In [12]:
# audio_row already set, just write meta_train.parquet
required_cols = ['audio_row', 'group_key', 'dataset', 'coarse_class']
extra_cols = [c for c in [
    'global_idx', 'wav_path', 'offset_s',
    'label', 'label_t2', 'label_t3', 'label_t4',
    'region', 'environment', 'source_file', 'window_idx',
] if c in meta_combined.columns]

meta_out = meta_combined[required_cols + extra_cols].copy()

assert (meta_out['audio_row'].values == np.arange(len(meta_out))).all(), \
    "audio_row not aligned"
assert meta_out['group_key'].notna().all(), "group_key has nulls"
assert meta_out['coarse_class'].notna().all(), "coarse_class has nulls"
assert len(meta_out) == X_audio_r.shape[0], \
    f"length mismatch: meta {len(meta_out)} vs X_audio {X_audio_r.shape[0]}"

META_TRAIN_OUT = OUT_DIR / 'meta_train.parquet'
meta_out.to_parquet(META_TRAIN_OUT, index=False)
print(f"Written: {META_TRAIN_OUT}")
print(f"  Rows    : {len(meta_out):,}")
print(f"  Columns : {list(meta_out.columns)}")
print("\nFinal class distribution:")
for cls, n in meta_out['coarse_class'].value_counts().items():
    print(f"  {cls:40s}  {n:>6,}")
print("\nFinal dataset distribution:")
for ds, n in meta_out['dataset'].value_counts().items():
    print(f"  {ds:25s}  {n:>6,}")
print("\nReady for cet_perch_finetune_run.ipynb ✓")

Written: /data2/mromaniuc/cet-det/tursiops_perch/meta_train.parquet
  Rows    : 65,256
  Columns : ['audio_row', 'group_key', 'dataset', 'coarse_class', 'global_idx', 'wav_path', 'offset_s', 'label', 'label_t2', 'label_t3', 'label_t4', 'region', 'environment', 'source_file', 'window_idx']

Final class distribution:
  background                                58,338
  Tursiops_truncatus                         6,918

Final dataset distribution:
  ECOSS_testtrain            35,936
  OLTREMARE                  16,860
  DRYAD                       6,770
  ALNITAK_CAVANILLES          4,406
  Adriatic_Sea                1,180
  ECOSS_enhanced                 52
  MONISH                         28
  WATKINS                        24

Ready for cet_perch_finetune_run.ipynb ✓


## 10. Final compatibility check with fine-tuning notebook

In [14]:
print("Final compatibility check...")

X_check    = np.load(str(AUDIO_CACHE), mmap_mode='r')
meta_check = pd.read_parquet(META_TRAIN_OUT)  # ← was META_OUT

assert X_check.shape[1] == AUDIO_LEN, \
    f"X_audio width {X_check.shape[1]} != {AUDIO_LEN}"
assert X_check.dtype == np.float32, \
    f"X_audio dtype {X_check.dtype} != float32"
assert len(meta_check) == X_check.shape[0], \
    f"Length mismatch: meta {len(meta_check)} vs X_audio {X_check.shape[0]}"
assert (meta_check['audio_row'].values == np.arange(len(meta_check))).all(), \
    "audio_row not aligned with row position"
assert set(required_cols).issubset(meta_check.columns), \
    f"Missing required columns: {set(required_cols) - set(meta_check.columns)}"

print("  X_audio.npy          OK")
print("  meta_train.parquet   OK")
print(f"  N windows            {X_check.shape[0]:,}")
print(f"  Classes              {sorted(meta_check['coarse_class'].unique())}")
print(f"  N datasets           {meta_check['dataset'].nunique()}")
print(f"  N groups             {meta_check['group_key'].nunique():,}")
print()
print("Ready for cet_perch_finetune_run.ipynb  ✓")

Final compatibility check...
  X_audio.npy          OK
  meta_train.parquet   OK
  N windows            65,256
  Classes              ['Tursiops_truncatus', 'background']
  N datasets           8
  N groups             1,500

Ready for cet_perch_finetune_run.ipynb  ✓
